Options:

--page_range TEXT: Specify which pages to process. Accepts comma-separated page numbers and ranges. Example: --page_range "0,5-10,20" will process pages 0, 5 through 10, and page 20.
--output_format [markdown|json|html|chunks]: Specify the format for the output results.
--output_dir PATH: Directory where output files will be saved. Defaults to the value specified in settings.OUTPUT_DIR.
--paginate_output: Paginates the output, using \n\n{PAGE_NUMBER} followed by - * 48, then \n\n
--use_llm: Uses an LLM to improve accuracy. You will need to configure the LLM backend - see below.
--format_lines: Reformat all lines using a local OCR model (inline math, underlines, bold, etc.). This will give very good quality math output.
--force_ocr: Force OCR processing on the entire document, even for pages that might contain extractable text.
--block_correction_prompt: if LLM mode is active, an optional prompt that will be used to correct the output of marker. This is useful for custom formatting or logic that you want to apply to the output.
--strip_existing_ocr: Remove all existing OCR text in the document and re-OCR with surya.
--redo_inline_math: If you want the absolute highest quality inline math conversion, use this along with --use_llm.
--disable_image_extraction: Don't extract images from the PDF. If you also specify --use_llm, then images will be replaced with a description.
--debug: Enable debug mode for additional logging and diagnostic information.
--processors TEXT: Override the default processors by providing their full module paths, separated by commas. Example: --processors "module1.processor1,module2.processor2"
--config_json PATH: Path to a JSON configuration file containing additional settings.
config --help: List all available builders, processors, and converters, and their associated configuration. These values can be used to build a JSON configuration file for additional tweaking of marker defaults.
--converter_cls: One of marker.converters.pdf.PdfConverter (default) or marker.converters.table.TableConverter. The PdfConverter will convert the whole PDF, the TableConverter will only extract and convert tables.
--llm_service: Which llm service to use if --use_llm is passed. This defaults to marker.services.gemini.GoogleGeminiService.
--help: see all of the flags that can be passed into marker. (it supports many more options then are listed above)
The list of supported languages for surya OCR is here. If you don't need OCR, marker can work with any language.

In [1]:
# base setting
# this takes approx. 5 mins to load

from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.output import text_from_rendered

converter = PdfConverter(
    artifact_dict=create_model_dict(),
)

# save both a markdown and a json file

c:\Users\dand\Desktop\Desktop\venvparse\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# this took 13mins to run for 1 document

filepath = r"C:\Users\dand\Desktop\Desktop\SDG-synthesis-coalition-tidy\1405_1.pdf"


rendered = converter(filepath)
text, _, images = text_from_rendered(rendered)

Recognizing tables: 100%|██████████| 6/6 [02:22<00:00, 23.69s/it]
Detecting bboxes: 0it [00:00, ?it/s]


In [ ]:
filepath = r"C:\Users\dand\Desktop\Desktop\SDG-synthesis-coalition-tidy\491_1.pdf"

rendered = converter(filepath)
# text, _, images = text_from_rendered(rendered)

In [3]:
# export the markdown file
def save_as_markdown(filepath,data):
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(data)

In [5]:
import os
output_folder = r"C:\Users\dand\Desktop\Desktop\SDG-synthesis-coalition-tidy\marker-parsed\markdown"
os.makedirs(output_folder, exist_ok=True)
filename="1405_1.pdf"
output_name = os.path.splitext(filename)[0] + ".md"
output_path = os.path.join(output_folder, output_name)

save_as_markdown(output_path,rendered.markdown)

In [ ]:
# parse the first 500 documents and extract markdown
# figure out json extraction
# qualitative evaluation of a random 10 documents
# filtering documents

In [ ]:
import json

with open('output.json', 'w', encoding='utf-8') as f:
    json.dump(rendered.model_dump_json(), f, indent=2)

# this output is way messier than the markdown output, would need to figure out which fields we're looking for
# alternatively, can we work with markdown? 

In [ ]:
pdf_folder = r"C:\Users\dand\Desktop\Desktop\SDG-synthesis-coalition-tidy"
output_folder = r"C:\Users\dand\Desktop\Desktop\SDG-synthesis-coalition-tidy\marker-parsed\markdown"

os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(pdf_folder):
    if not filename.lower().endswith(".pdf"):
        continue  # skip non-PDF files

    input_path = os.path.join(pdf_folder, filename)
    output_name = os.path.splitext(filename)[0] + ".md"
    output_path = os.path.join(output_folder, output_name)

    try:
        rendered = converter(input_path)
        save_as_markdown(output_path, rendered.markdown)
        print(f" Processed: {filename}")
    except Exception as e:
        print(f" Error processing {filename}: {e}")